In [1]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\Users\Gulso\Documents\Gul\CredBlock-FL\.venv\Scripts\python.exe


In [2]:
import pandas as pd

DATA_PATH = "../data/rba-dataset.csv"

# Read only the first 100,000 login records
df = pd.read_csv(DATA_PATH, nrows=100_000)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (100000, 16)


,index,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,OS Name and Version,Device Type,Login Successful,Is Attack IP,Is Account Takeover
0,0,2020-02-03 12:43:30.772,-4324475583306591935,NaN,10.0.65.171,NO,-,-,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Firefox 20.0.0.1618,iOS 13.4,mobile,False,False,False
1,1,2020-02-03 12:43:43.549,-4324475583306591935,NaN,194.87.207.6,AU,-,-,60117,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile 46.0.2490,Android 4.1,mobile,False,False,False
2,2,2020-02-03 12:43:55.873,-3284137479262433373,NaN,81.167.144.58,NO,Vestland,Urangsvag,29695,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,iOS 7.1,mobile,True,False,False
3,3,2020-02-03 12:43:56.180,-4324475583306591935,NaN,170.39.78.152,US,-,-,393398,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile WebView 85.0.4183,Android 4.1,mobile,False,False,False
4,4,2020-02-03 12:43:59.396,-4618854071942621186,NaN,10.0.0.47,US,Virginia,Ashburn,398986,Mozilla/5.0 (Linux; U; Android 2.2) Build/NMA...,Chrome Mobile WebView 85.0.4183,Android 2.2,mobile,False,True,False


In [3]:
# Basic information about the 100,000-row sample

print("Total login records:", len(df))
print("Unique users:", df["User ID"].nunique())
print("Unique IP addresses:", df["IP Address"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

print("\nLogin Successful:")
print(df["Login Successful"].value_counts())

print("\nAttack IP:")
print(df["Is Attack IP"].value_counts())

print("\nAccount Takeover:")
print(df["Is Account Takeover"].value_counts())

Total login records: 100000
Unique users: 41500
Unique IP addresses: 45437

Missing values:
index                           0
Login Timestamp                 0
User ID                         0
Round-Trip Time [ms]        95718
IP Address                      0
Country                         0
Region                         78
City                           35
ASN                             0
User Agent String               0
Browser Name and Version        0
OS Name and Version             0
Device Type                     7
Login Successful                0
Is Attack IP                    0
Is Account Takeover             0
dtype: int64

Login Successful:
Login Successful
False    51273
True     48727
Name: count, dtype: int64

Attack IP:
Is Attack IP
False    90302
True      9698
Name: count, dtype: int64

Account Takeover:
Is Account Takeover
False    99998
True         2
Name: count, dtype: int64


In [4]:
def show_distribution(column):
    counts = df[column].value_counts()
    percentages = df[column].value_counts(normalize=True) * 100

    result = pd.DataFrame({
        "Count": counts,
        "Percentage (%)": percentages.round(3)
    })

    print(f"\n{column}")
    print(result)


show_distribution("Login Successful")
show_distribution("Is Attack IP")
show_distribution("Is Account Takeover")


Login Successful
                  Count  Percentage (%)
Login Successful                       
False             51273          51.273
True              48727          48.727

Is Attack IP
              Count  Percentage (%)
Is Attack IP                       
False         90302          90.302
True           9698           9.698

Is Account Takeover
                     Count  Percentage (%)
Is Account Takeover                       
False                99998          99.998
True                     2           0.002


In [5]:
with open("../data/README.md", "r", encoding="utf-8") as file:
    readme = file.read()

print(readme)

# Login Data Set for Risk-Based Authentication

> Synthesized login feature data of >33M login attempts and >3.3M users
> on a large-scale online service in Norway. Original data collected
> between February 2020 and February 2021.

This data sets aims to foster research and development for
[Risk-Based Authentication (RBA)] systems. The data was synthesized from
the real-world login behavior of more than 3.3M users at a large-scale
single sign-on (SSO) online service in Norway.

The users used this SSO to access sensitive data provided by the online
service, e.g., a cloud storage and billing information. We used this
data set to study how the [Freeman et al. (2016)] RBA model behaves on a
large-scale online service in the real world (see [Publication](#publication)). The
synthesized data set can reproduce these results made on the original
data set (see [Study Reproduction](#study-reproduction)). Beyond that, you can use this data
set to evaluate and improve RBA algorithms under real-w

In [6]:
terms = [
    "Login Successful",
    "Is Attack IP",
    "Is Account Takeover"
]

lines = readme.splitlines()

for term in terms:
    print(f"\n===== {term} =====")

    for i, line in enumerate(lines):
        if term.lower() in line.lower():
            start = max(0, i - 3)
            end = min(len(lines), i + 6)

            for nearby_line in lines[start:end]:
                print(nearby_line)


===== Login Successful =====
User ID                    | Integer   | Idenfication number related to the affected user account                                         | [Random pseudonym]
Login Timestamp            | Integer   | Timestamp related to the login attempt                                                           | [64 Bit timestamp]
Round-Trip Time (RTT) [ms] | Integer   | Server-side measured latency between client and server                                           | 1 - 8600000
Login Successful           | Boolean   | `True`: Login was successful, `False`: Login failed                                              | (`true`, `false`)
Is Attack IP               | Boolean   | IP address was found in known attacker data set                                                  | (`true`, `false`)
Is Account Takeover        | Boolean   | Login attempt was identified as account takeover by incident response team of the online service | (`true`, `false`)

[^1]: Few (invalid) user 

In [7]:
# Process the full RBA dataset in manageable chunks

CHUNK_SIZE = 500_000

total_rows = 0
login_success_counts = {}
attack_ip_counts = {}
takeover_counts = {}

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    usecols=[
        "Login Successful",
        "Is Attack IP",
        "Is Account Takeover"
    ]
)

for chunk_number, chunk in enumerate(reader, start=1):
    total_rows += len(chunk)

    # Count each True/False label in this chunk
    for column, storage in [
        ("Login Successful", login_success_counts),
        ("Is Attack IP", attack_ip_counts),
        ("Is Account Takeover", takeover_counts)
    ]:
        counts = chunk[column].value_counts()

        for value, count in counts.items():
            storage[value] = storage.get(value, 0) + count

    print(f"Processed chunk {chunk_number}: {total_rows:,} rows")

print("\n--- FULL DATASET RESULTS ---")
print("Total rows:", f"{total_rows:,}")
print("Login Successful:", login_success_counts)
print("Is Attack IP:", attack_ip_counts)
print("Is Account Takeover:", takeover_counts)

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14

In [8]:
# Count missing values across the complete RBA dataset

CHUNK_SIZE = 500_000

missing_counts = None
total_rows = 0

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE
)

for chunk_number, chunk in enumerate(reader, start=1):
    
    chunk_missing = chunk.isnull().sum()

    if missing_counts is None:
        missing_counts = chunk_missing
    else:
        missing_counts = missing_counts.add(chunk_missing, fill_value=0)

    total_rows += len(chunk)

    print(f"Processed chunk {chunk_number}: {total_rows:,} rows")


missing_percentages = (missing_counts / total_rows * 100).round(3)

missing_results = pd.DataFrame({
    "Missing Count": missing_counts.astype(int),
    "Missing Percentage (%)": missing_percentages
})

print("\n--- FULL DATASET MISSING VALUES ---")
print(missing_results)

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14

In [9]:
# Count unique users and IP addresses across the full dataset

CHUNK_SIZE = 500_000

unique_users = set()
unique_ips = set()
total_rows = 0

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    usecols=["User ID", "IP Address"]
)

for chunk_number, chunk in enumerate(reader, start=1):

    unique_users.update(chunk["User ID"].unique())
    unique_ips.update(chunk["IP Address"].unique())

    total_rows += len(chunk)

    print(f"Processed chunk {chunk_number}: {total_rows:,} rows")

print("\n--- FULL DATASET GRAPH ENTITIES ---")
print("Total login records:", f"{total_rows:,}")
print("Unique users:", f"{len(unique_users):,}")
print("Unique IP addresses:", f"{len(unique_ips):,}")
print("Potential User + IP nodes:", f"{len(unique_users) + len(unique_ips):,}")

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14

In [10]:
# Count unique User-IP pairs across the full dataset

CHUNK_SIZE = 500_000

unique_pairs = set()
total_rows = 0

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    usecols=["User ID", "IP Address"]
)

for chunk_number, chunk in enumerate(reader, start=1):

    pairs = zip(chunk["User ID"], chunk["IP Address"])
    unique_pairs.update(pairs)

    total_rows += len(chunk)

    print(f"Processed chunk {chunk_number}: {total_rows:,} rows")

print("\n--- USER-IP RELATIONSHIP RESULTS ---")
print("Total login attempts:", f"{total_rows:,}")
print("Unique User-IP pairs:", f"{len(unique_pairs):,}")
print(
    "Repeated User-IP interactions:",
    f"{total_rows - len(unique_pairs):,}"
)
print(
    "Unique-pair ratio:",
    f"{(len(unique_pairs) / total_rows) * 100:.3f}%"
)

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14

In [11]:
import pandas as pd

DATA_PATH = "../data/rba-dataset.csv"
CHUNK_SIZE = 500_000

user_login_counts = {}

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["User ID"],
    chunksize=CHUNK_SIZE
):
    counts = chunk["User ID"].value_counts()

    for user_id, count in counts.items():
        user_login_counts[user_id] = (
            user_login_counts.get(user_id, 0) + count
        )

login_counts = pd.Series(user_login_counts)

print("--- LOGINS PER USER ---")
print("Total unique users:", len(login_counts))
print("Minimum logins per user:", login_counts.min())
print("Maximum logins per user:", login_counts.max())
print("Mean logins per user:", round(login_counts.mean(), 2))
print("Median logins per user:", login_counts.median())

print("\nPercentiles:")
print(login_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

--- LOGINS PER USER ---
Total unique users: 4304857
Minimum logins per user: 1
Maximum logins per user: 14025899
Mean logins per user: 7.26
Median logins per user: 2.0

Percentiles:
0.25     1.0
0.50     2.0
0.75     4.0
0.90     9.0
0.95    13.0
0.99    28.0
dtype: float64


In [12]:
print("--- TOP 10 USERS BY LOGIN COUNT ---")
print(login_counts.nlargest(10))

print("\nPercentage of all logins:")
top_users = login_counts.nlargest(10)

for user_id, count in top_users.items():
    percentage = (count / 31_269_264) * 100
    print(
        f"User {user_id}: "
        f"{count:,} logins ({percentage:.3f}%)"
    )

--- TOP 10 USERS BY LOGIN COUNT ---
-4324475583306591935    14025899
 6998943612473066845       70028
-7198559811247368245        6417
-4652302824165996131        1927
-2894367365566592989        1613
-3031272205102765641        1537
 4588325308801696666        1510
 9009909461743844032        1475
-7955135358862007287        1423
 8306338755768072775        1367
dtype: int64

Percentage of all logins:
User -4324475583306591935: 14,025,899 logins (44.855%)
User 6998943612473066845: 70,028 logins (0.224%)
User -7198559811247368245: 6,417 logins (0.021%)
User -4652302824165996131: 1,927 logins (0.006%)
User -2894367365566592989: 1,613 logins (0.005%)
User -3031272205102765641: 1,537 logins (0.005%)
User 4588325308801696666: 1,510 logins (0.005%)
User 9009909461743844032: 1,475 logins (0.005%)
User -7955135358862007287: 1,423 logins (0.005%)
User 8306338755768072775: 1,367 logins (0.004%)


In [13]:
TARGET_USER = -4324475583306591935
CHUNK_SIZE = 500_000

target_chunks = []

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE
):
    matches = chunk[chunk["User ID"] == TARGET_USER]

    if not matches.empty:
        target_chunks.append(matches)

target_df = pd.concat(target_chunks, ignore_index=True)

print("--- HIGH-ACTIVITY USER ANALYSIS ---")
print("Login records:", f"{len(target_df):,}")
print("Unique IPs:", f"{target_df['IP Address'].nunique():,}")

print("\nLogin Successful:")
print(target_df["Login Successful"].value_counts())

print("\nAttack IP:")
print(target_df["Is Attack IP"].value_counts())

print("\nAccount Takeover:")
print(target_df["Is Account Takeover"].value_counts())

timestamps = pd.to_datetime(target_df["Login Timestamp"])

print("\nFirst login:", timestamps.min())
print("Last login:", timestamps.max())

print("\nTop 10 IPs:")
print(target_df["IP Address"].value_counts().head(10))

--- HIGH-ACTIVITY USER ANALYSIS ---
Login records: 14,025,899
Unique IPs: 2,047,733

Login Successful:
Login Successful
False    14025895
True            4
Name: count, dtype: int64

Attack IP:
Is Attack IP
False    12375272
True      1650627
Name: count, dtype: int64

Account Takeover:
Is Account Takeover
False    14025899
Name: count, dtype: int64

First login: 2020-02-03 12:43:30.772000
Last login: 2021-02-28 23:59:56.343000

Top 10 IPs:
IP Address
10.3.205.197     444433
23.137.225.33    321325
10.3.205.196     257343
10.3.205.195     236902
10.3.205.194     178138
10.3.205.193     152262
10.3.205.192      99589
10.3.205.191      88916
10.0.181.232      77747
170.39.78.106     67563
Name: count, dtype: int64


In [14]:
from pathlib import Path

print("Notebook working directory:")
print(Path.cwd())

data_dir = Path("../data").resolve()

print("\nExpected data folder:")
print(data_dir)

print("\nDoes data folder exist?")
print(data_dir.exists())

print("\nMatching summary files:")
for file in data_dir.glob("rba_user_summary*"):
    print(file)

Notebook working directory:
c:\Users\Gulso\Documents\Gul\CredBlock-FL\notebooks

Expected data folder:
C:\Users\Gulso\Documents\Gul\CredBlock-FL\data

Does data folder exist?
True

Matching summary files:
C:\Users\Gulso\Documents\Gul\CredBlock-FL\data\rba_user_summary.parquet


In [15]:
from pathlib import Path
import duckdb

DATA_PATH = Path("../data/rba-dataset.csv").resolve()
OUTPUT_PATH = Path("../data/rba_user_summary.parquet").resolve()

OUTLIER_USER = -4324475583306591935

data_path = DATA_PATH.as_posix()
output_path = OUTPUT_PATH.as_posix()

print("Reading from:")
print(DATA_PATH)

print("\nWriting to:")
print(OUTPUT_PATH)

query = f"""
COPY (
    SELECT
        "User ID" AS user_id,
        COUNT(*) AS total_logins,

        SUM(CASE
            WHEN "Login Successful" = TRUE
            THEN 1 ELSE 0
        END) AS successful_logins,

        SUM(CASE
            WHEN "Login Successful" = FALSE
            THEN 1 ELSE 0
        END) AS failed_logins,

        SUM(CASE
            WHEN "Is Attack IP" = TRUE
            THEN 1 ELSE 0
        END) AS attack_ip_logins,

        SUM(CASE
            WHEN "Is Account Takeover" = TRUE
            THEN 1 ELSE 0
        END) AS account_takeover_logins

    FROM read_csv_auto('{data_path}')

    WHERE "User ID" != {OUTLIER_USER}

    GROUP BY "User ID"
)
TO '{output_path}'
(FORMAT PARQUET);
"""

duckdb.sql(query)

print("\nFinished.")
print("File created:", OUTPUT_PATH.exists())

if OUTPUT_PATH.exists():
    print(
        "File size:",
        round(OUTPUT_PATH.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

Reading from:
C:\Users\Gulso\Documents\Gul\CredBlock-FL\data\rba-dataset.csv

Writing to:
C:\Users\Gulso\Documents\Gul\CredBlock-FL\data\rba_user_summary.parquet

Finished.
File created: True
File size: 42.3 MB


In [16]:
import duckdb

SUMMARY_PATH = "../data/rba_user_summary.parquet"

print("--- FIRST 5 USERS ---")
display(
    duckdb.sql(f"""
        SELECT *
        FROM read_parquet('{SUMMARY_PATH}')
        LIMIT 5
    """).df()
)

print("\n--- SUMMARY CHECK ---")

summary_check = duckdb.sql(f"""
    SELECT
        COUNT(*) AS users,
        SUM(total_logins) AS login_records,
        SUM(successful_logins) AS successful_logins,
        SUM(failed_logins) AS failed_logins,
        SUM(attack_ip_logins) AS attack_ip_logins,
        SUM(account_takeover_logins) AS account_takeover_logins
    FROM read_parquet('{SUMMARY_PATH}')
""").df()

display(summary_check)

--- FIRST 5 USERS ---


,user_id,total_logins,successful_logins,failed_logins,attack_ip_logins,account_takeover_logins
0,-2504424457940726811,1,1.0,0.0,0.0,0.0
1,4192498960432583471,30,21.0,9.0,14.0,0.0
2,1551700423135884095,12,7.0,5.0,4.0,0.0
3,-576882492626270429,1,1.0,0.0,0.0,0.0
4,7030068145062612522,13,0.0,13.0,0.0,0.0



--- SUMMARY CHECK ---


,users,login_records,successful_logins,failed_logins,attack_ip_logins,account_takeover_logins
0,4304856,17243365.0,12541438.0,4701927.0,1446350.0,141.0


In [17]:
security_profile = duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_users,

        COUNT(*) FILTER (
            WHERE successful_logins > 0
        ) AS users_with_successful_logins,

        COUNT(*) FILTER (
            WHERE failed_logins > 0
        ) AS users_with_failed_logins,

        COUNT(*) FILTER (
            WHERE attack_ip_logins > 0
        ) AS users_with_attack_ip_activity,

        COUNT(*) FILTER (
            WHERE account_takeover_logins > 0
        ) AS users_with_account_takeover,

        COUNT(*) FILTER (
            WHERE successful_logins = 0
        ) AS users_with_only_failed_logins

    FROM read_parquet('{SUMMARY_PATH}')
""").df()

display(security_profile)

,total_users,users_with_successful_logins,users_with_failed_logins,users_with_attack_ip_activity,users_with_account_takeover,users_with_only_failed_logins
0,4304856,3304501,2400076,742348,138,1000355


In [18]:
from pathlib import Path
import duckdb

SUMMARY_PATH = Path("../data/rba_user_summary.parquet").resolve()
ASSIGNMENT_PATH = Path("../data/rba_user_org_assignment.parquet").resolve()

summary_path = SUMMARY_PATH.as_posix()
assignment_path = ASSIGNMENT_PATH.as_posix()

query = f"""
COPY (
    WITH base AS (
        SELECT
            *,

            CASE
                WHEN account_takeover_logins > 0
                    THEN 'account_takeover'

                WHEN attack_ip_logins > 0
                    THEN 'attack_ip'

                WHEN successful_logins = 0
                    THEN 'failed_only'

                ELSE 'general'
            END AS stratum,

            hash(user_id) % 10000 AS bucket

        FROM read_parquet('{summary_path}')
    ),

    ranked AS (
        SELECT
            *,

            CASE
                WHEN stratum = 'account_takeover'
                THEN ROW_NUMBER() OVER (
                    PARTITION BY stratum
                    ORDER BY hash(user_id)
                )
            END AS ato_rank

        FROM base
    )

    SELECT
        user_id,
        total_logins,
        successful_logins,
        failed_logins,
        attack_ip_logins,
        account_takeover_logins,
        stratum,

        CASE

            -- Rare ATO users: distribute approximately evenly
            WHEN stratum = 'account_takeover' THEN
                'Org' || CAST(
                    (((ato_rank - 1) % 5) + 1)
                    AS VARCHAR
                )

            -- Attack-IP users:
            -- 16%, 18%, 20%, 22%, 24%
            WHEN stratum = 'attack_ip' THEN
                CASE
                    WHEN bucket < 1600 THEN 'Org1'
                    WHEN bucket < 3400 THEN 'Org2'
                    WHEN bucket < 5400 THEN 'Org3'
                    WHEN bucket < 7600 THEN 'Org4'
                    ELSE 'Org5'
                END

            -- Failed-only users:
            -- 22%, 21%, 20%, 19%, 18%
            WHEN stratum = 'failed_only' THEN
                CASE
                    WHEN bucket < 2200 THEN 'Org1'
                    WHEN bucket < 4300 THEN 'Org2'
                    WHEN bucket < 6300 THEN 'Org3'
                    WHEN bucket < 8200 THEN 'Org4'
                    ELSE 'Org5'
                END

            -- General users:
            -- 18%, 19%, 20%, 21%, 22%
            ELSE
                CASE
                    WHEN bucket < 1800 THEN 'Org1'
                    WHEN bucket < 3700 THEN 'Org2'
                    WHEN bucket < 5700 THEN 'Org3'
                    WHEN bucket < 7800 THEN 'Org4'
                    ELSE 'Org5'
                END

        END AS organization

    FROM ranked
)
TO '{assignment_path}'
(FORMAT PARQUET);
"""

duckdb.sql(query)

print("Organization assignment created.")
print("File:", ASSIGNMENT_PATH)
print("Exists:", ASSIGNMENT_PATH.exists())

Organization assignment created.
File: C:\Users\Gulso\Documents\Gul\CredBlock-FL\data\rba_user_org_assignment.parquet
Exists: True


In [19]:
org_summary = duckdb.sql(f"""
    SELECT
        organization,

        COUNT(*) AS users,

        SUM(total_logins) AS login_records,

        ROUND(
            100.0 * SUM(total_logins)
            / SUM(SUM(total_logins)) OVER (),
            2
        ) AS login_share_pct,

        ROUND(
            100.0 * SUM(successful_logins)
            / SUM(total_logins),
            2
        ) AS successful_pct,

        ROUND(
            100.0 * SUM(failed_logins)
            / SUM(total_logins),
            2
        ) AS failed_pct,

        ROUND(
            100.0 * SUM(attack_ip_logins)
            / SUM(total_logins),
            2
        ) AS attack_ip_pct,

        SUM(
            CASE
                WHEN account_takeover_logins > 0
                THEN 1 ELSE 0
            END
        ) AS ato_users

    FROM read_parquet('{assignment_path}')

    GROUP BY organization
    ORDER BY organization
""").df()

display(org_summary)

,organization,users,login_records,login_share_pct,successful_pct,failed_pct,attack_ip_pct,ato_users
0,Org1,791567,3080150.0,17.86,72.07,27.93,7.52,28.0
1,Org2,825185,3250845.0,18.85,72.59,27.41,7.95,28.0
2,Org3,861765,3520641.0,20.42,71.64,28.36,8.36,28.0
3,Org4,895714,3611088.0,20.94,73.43,26.57,8.77,27.0
4,Org5,930625,3780641.0,21.93,73.75,26.25,9.13,27.0


In [20]:
integrity_check = duckdb.sql(f"""
    SELECT
        COUNT(*) AS assignment_rows,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT organization) AS organizations,
        SUM(total_logins) AS total_logins
    FROM read_parquet('{assignment_path}')
""").df()

display(integrity_check)

,assignment_rows,unique_users,organizations,total_logins
0,4304856,4304856,5,17243365.0


In [21]:
from pathlib import Path
import shutil
import duckdb

DATA_PATH = Path("../data/rba-dataset.csv").resolve()
ASSIGNMENT_PATH = Path("../data/rba_user_org_assignment.parquet").resolve()
ORG_DATA_DIR = Path("../data/organizations").resolve()

# Remove old output if this cell is rerun
if ORG_DATA_DIR.exists():
    shutil.rmtree(ORG_DATA_DIR)

ORG_DATA_DIR.mkdir(parents=True, exist_ok=True)

data_path = DATA_PATH.as_posix()
assignment_path = ASSIGNMENT_PATH.as_posix()
org_data_path = ORG_DATA_DIR.as_posix()

print("Creating organization datasets...")
print("Output folder:", ORG_DATA_DIR)

duckdb.sql(f"""
    COPY (
        SELECT
            r.*,
            a.organization

        FROM read_csv_auto('{data_path}') AS r

        INNER JOIN read_parquet('{assignment_path}') AS a
            ON r."User ID" = a.user_id
    )

    TO '{org_data_path}'
    (
        FORMAT PARQUET,
        PARTITION_BY (organization),
        OVERWRITE_OR_IGNORE
    );
""")

print("\nFinished.")

Creating organization datasets...
Output folder: C:\Users\Gulso\Documents\Gul\CredBlock-FL\data\organizations

Finished.


In [22]:
ORG_GLOB = (
    Path("../data/organizations")
    .resolve()
    .as_posix()
    + "/**/*.parquet"
)

org_dataset_check = duckdb.sql(f"""
    SELECT
        organization,
        COUNT(*) AS login_records,
        COUNT(DISTINCT "User ID") AS unique_users

    FROM read_parquet(
        '{ORG_GLOB}',
        hive_partitioning = true
    )

    GROUP BY organization
    ORDER BY organization
""").df()

display(org_dataset_check)

print(
    "\nTotal records:",
    f"{org_dataset_check['login_records'].sum():,}"
)

,organization,login_records,unique_users
0,Org1,3080150,791567
1,Org2,3250845,825185
2,Org3,3520641,861765
3,Org4,3611088,895714
4,Org5,3780641,930625



Total records: 17,243,365
